# Track A: is linear attention's associative matrix a useful store?

Store N items (key vector → item id) in one fixed d×d matrix, look them up by a query, and compare with FAISS. Every compressed method gets **the same bytes** as the matrix; product quantisation (PQ) at equal memory is the fair competitor. Protocol, decision rules and predictions: `memtest/PROTOCOL.md` (written before any run).

| stage | what | rough time |
|---|---|---|
| A1 capacity | random keys, d = 64 / 256 / 1024, N from d/8 to 8d, 5 seeds | longest stage (delta-rule loops at d = 1024) |
| A2 real data | Quora duplicate questions, MiniLM-L6 embeddings, 3 budgets, dev then test | minutes |
| A3 updates | overwrite / delete items, stale answers, float drift | minutes |
| A5 cost | write and read time, N up to 300,000 | minutes |

**Runtime:** CPU is enough; a T4 only speeds up the embedding step. Times are not measured yet.

Each stage writes a `.done` marker, so after a disconnect re-run all cells and finished stages are skipped (set `RESTORE = True` and upload `results.zip` if the runtime was deleted).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/store.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

In [ ]:
!pip install -q faiss-cpu sentence-transformers

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
EMB = '/content/data/quora_emb.npz'
def stage(name, args):
    marker = f'{OUTDIR}/store_{name}.done'
    if os.path.exists(marker):
        print('skip', name, '(done)'); return
    subprocess.run([sys.executable, '-m', 'memtest.store', *args, '--out', RUNS, '--emb', EMB], cwd=REPO, check=True)
    open(marker, 'w').close()

Embed the Quora pairs once (dedupe, split into calibration / dev / test, 20,000 pairs each).

In [ ]:
if not os.path.exists(EMB):
    subprocess.run([sys.executable, '-m', 'memtest.store', 'embed', '--emb', EMB], cwd=REPO, check=True)

## A1: capacity on random keys

In [ ]:
stage('capacity', ['capacity'])

## A2: real data. Dev first (the matrix variant is chosen here), then test (used once).

In [ ]:
stage('real_dev', ['real', '--split', 'dev'])

In [ ]:
stage('real_test', ['real', '--split', 'test'])

## A3: updates and deletes

In [ ]:
stage('updates', ['updates'])

## A5: cost against N

In [ ]:
stage('cost', ['cost'])

## Results

In [ ]:
out = subprocess.run([sys.executable, '-m', 'memtest.store', 'summary', '--out', RUNS], cwd=REPO,
                     capture_output=True, text=True, check=True).stdout
open(f'{OUTDIR}/store_summary.md', 'w').write(out)
from IPython.display import Markdown, display
display(Markdown(out))

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from collections import defaultdict
rows = [json.loads(l) for l in open(RUNS) if '"exp": "real"' in l]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, dv in zip(axes, (64, 384, 1536)):
    by = defaultdict(list)
    for r in rows:
        if r['split'] == 'test' and r['dv'] == dv and r['recall1'] is not None:
            by[r['method']].append((r['N'], r['recall1']))
    for m in ('add+c', 'delta+c', 'lstsq+c', 'pq', 'fifo', 'flat'):
        if by[m]:
            Ns = sorted({n for n, _ in by[m]})
            ax.plot(Ns, [100 * np.mean([v for n, v in by[m] if n == N]) for N in Ns], marker='o', label=m)
    ax.set_xscale('log'); ax.set_title(f'test, {384 * dv * 4 / 1e3:.0f} KB'); ax.set_xlabel('items stored (N)')
    ax.grid(alpha=.3)
axes[0].set_ylabel('recall@1 %'); axes[0].legend()
plt.savefig(f'{OUTDIR}/store_real.png', dpi=150, bbox_inches='tight'); plt.show()

## How to read it (decision rules from `memtest/PROTOCOL.md`)

- **Dropped as a store** if the chosen matrix variant is worse than PQ at N = 384 and N = 768 (Holm p < 0.05).
- **"Useful memory tier"** only if it is within 2 points of PQ, or better, at every N ≥ 384.
- `flat` and `hnsw` have no budget: they show the best any method could do, not a fair comparison.

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')